# Tidying a Data Set

This notebook uses the flattened temperature data for San Francisco airport, to produce a tidy data set of **daily average** temperatures in that location.

### Steps below:

1. Connect to Snowflake
2. Access the temperature table
3. Tidy the data
4. Save the result

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.functions import *
from snowflake.snowpark.session import Session
from snowflake.snowpark.types import *

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

### 1. Connect to Snowflake

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], 'rb') as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{'private_key': private_key_bytes}}).create()

### 2. Access the temperature table

In [ ]:
sfo_tempsDF = session.table('public.sfo_temps')

* Examine the table

In [ ]:
sfo_tempsDF.schema.fields

In [ ]:
sfo_tempsDF.count()

In [ ]:
sfo_tempsDF.show(5)

* Notice the extreme values of air temperature in the data

In [ ]:
(sfo_tempsDF.
    agg([min(col('air_temp')), 
         max(col('air_temp'))]).
    show()
)

An air temperature of 999.9 occurs when the data collector failed to record a valid measure. How many times does this "sentinel" value occur? 

In [ ]:
(sfo_tempsDF.
    filter(col('air_temp')>999.0).
    count()
)

Obviously, these sentinel values of 999.9 must not be included in any calculation of daily average temperatures: we'll eliminate them while tidying the data set.

### 3. Tidy the data

It's easy to chain together a sequence of data tranformations using DataFrame methods.

In [ ]:
sfo_daily_tempsDF = (
    sfo_tempsDF.
    filter(col('air_temp') < 999).                  # Remove rows with sentinel value 999.9
    withColumn('date', col('dt').cast(DateType())). # Extract date alone from DateTime column
    select('date', 'air_temp').                     # Select only the two columns
    groupBy(col('date')).avg(col('air_temp')).      # Get average temperature for each date
    rename(col('AVG(AIR_TEMP)'), 'daily_avg_temp')  # Rename the computed average column
)

* Examine the tidy DataFrame

In [ ]:
sfo_daily_tempsDF.schema.fields

In [ ]:
sfo_daily_tempsDF.count()

Look at the first few and the last few dates represented in the data.

In [ ]:
sfo_daily_tempsDF.sort(col('date')).show(5)

In [ ]:
sfo_daily_tempsDF.sort(col('date').desc()).show(5)

* Verify the minimum and maximum temperatures

In [ ]:
(sfo_daily_tempsDF.
    agg([min(col('daily_avg_temp')), 
         max(col('daily_avg_temp'))]).
    show()
)

### 4. Save the result

In [ ]:
sfo_daily_tempsDF.write.mode('overwrite').saveAsTable('public.sfo_daily_temps')